
# Pertemuan 06 — Binary Search: Deep Dive & Studi Kasus

**Mata Kuliah:** Algoritma dan Pemrograman  
**Fokus:** Pendalaman Binary Search, analisis efisiensi, dan pemilihan algoritma berdasarkan karakteristik data.

Notebook ini dirancang sebagai *guided exploration*: mahasiswa tidak hanya menyalin kode, tetapi menjalankan eksperimen, membaca jejak proses pencarian, membandingkan pendekatan, lalu menganalisis studi kasus.

---

## Capaian Pembelajaran Pertemuan

Setelah menyelesaikan notebook ini, mahasiswa diharapkan mampu:

1. Menjelaskan cara kerja Binary Search menggunakan konsep `low`, `mid`, dan `high`.
2. Menjelaskan **syarat utama** penggunaan Binary Search.
3. Membandingkan Linear Search dan Binary Search dari sisi jumlah operasi dan kompleksitas waktu.
4. Menerapkan Binary Search pada beberapa studi kasus.
5. Menentukan kapan Binary Search **cocok** dan **tidak cocok** digunakan.
6. Mengenal variasi Binary Search seperti *lower bound*, *upper bound*, pencarian nilai terdekat, dan *Binary Search on Answer*.



## 1. Fenomena: Bagaimana Mencari 1 Data dari 1.000.000 Data?

Bayangkan sebuah sistem memiliki **1.000.000 ID mahasiswa** yang sudah terurut.

Kita ingin mencari:

```text
583421
```

Pertanyaannya:

> Apakah komputer harus memeriksa data satu per satu dari awal?

Mari mulai dari pendekatan paling sederhana: **Linear Search**.


In [ ]:

def linear_search(data, target):
    comparisons = 0

    for i, value in enumerate(data):
        comparisons += 1

        if value == target:
            return i, comparisons

    return -1, comparisons


data = list(range(1, 1_000_001))
target = 583_421

index, comparisons = linear_search(data, target)

print("Index ditemukan :", index)
print("Jumlah pemeriksaan:", comparisons)



### Diskusi

1. Berapa kali perbandingan dilakukan?
2. Bagaimana jika data yang dicari berada pada posisi terakhir?
3. Bagaimana jika data tidak ditemukan?
4. Jika jumlah data bertambah menjadi 10 juta, apa yang terjadi?

Linear Search memiliki kompleksitas waktu:

\[
O(n)
\]

Dalam *worst case*, jumlah data yang diperiksa dapat mendekati `n`.



## 2. Ide Utama Binary Search

Jika data **sudah terurut**, kita tidak perlu memeriksa semua elemen.

Binary Search bekerja dengan cara:

1. Lihat elemen di tengah.
2. Bandingkan dengan target.
3. Jika target lebih kecil → buang setengah bagian kanan.
4. Jika target lebih besar → buang setengah bagian kiri.
5. Ulangi sampai data ditemukan atau ruang pencarian habis.

> **Intuisi penting:** setiap langkah Binary Search menghilangkan kira-kira setengah ruang pencarian.



## 3. Syarat Binary Search

Binary Search paling mudah digunakan ketika:

- data **terurut berdasarkan key yang dicari**;
- data mendukung akses ke elemen tengah secara efisien;
- pencarian dilakukan cukup sering sehingga struktur terurut memberi manfaat.

Contoh:

```python
[3, 7, 11, 18, 25, 31, 42, 56, 68, 79, 91]
```

Target:

```text
68
```

Pada langkah pertama, nilai tengah adalah `31`.

Karena:

```text
68 > 31
```

maka semua nilai di sebelah kiri `31` dapat diabaikan.


In [ ]:

def binary_search_trace(data, target):
    low = 0
    high = len(data) - 1
    step = 1

    while low <= high:
        mid = (low + high) // 2

        print(
            f"Step {step}: "
            f"low={low}, mid={mid}, high={high}, "
            f"value={data[mid]}"
        )

        if data[mid] == target:
            print(f"✅ Target {target} ditemukan pada index {mid}")
            return mid

        elif data[mid] < target:
            print("   Target lebih besar → cari di kanan")
            low = mid + 1

        else:
            print("   Target lebih kecil → cari di kiri")
            high = mid - 1

        step += 1

    print("❌ Target tidak ditemukan")
    return -1


data = [3, 7, 11, 18, 25, 31, 42, 56, 68, 79, 91]
binary_search_trace(data, 68)



### Perhatikan perubahan `low`, `mid`, dan `high`

Pada setiap iterasi, ruang pencarian semakin kecil.

Rumus indeks tengah:

```python
mid = (low + high) // 2
```

Kondisi penghentian:

```python
while low <= high:
```

Jika `low > high`, berarti tidak ada lagi ruang pencarian yang tersisa.


In [ ]:

def binary_search(data, target):
    low = 0
    high = len(data) - 1

    while low <= high:
        mid = (low + high) // 2

        if data[mid] == target:
            return mid

        elif data[mid] < target:
            low = mid + 1

        else:
            high = mid - 1

    return -1


numbers = [3, 7, 11, 18, 25, 31, 42, 56, 68, 79, 91]

print(binary_search(numbers, 68))
print(binary_search(numbers, 100))



## 4. Mengapa Kompleksitasnya `O(log n)`?

Misalkan kita memiliki `1.000.000` data.

Binary Search membagi ruang pencarian menjadi dua secara berulang:

```text
1.000.000
500.000
250.000
125.000
62.500
...
1
```

Secara kasar, jumlah langkah maksimum adalah:

\[
\log_2(n)
\]

Untuk 1.000.000 data, hanya sekitar **20 langkah**.


In [ ]:

import math

sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]

print(f"{'Jumlah Data':>15} | {'Linear Worst Case':>18} | {'Binary ~ log2(n)':>18}")
print("-" * 60)

for n in sizes:
    linear_steps = n
    binary_steps = math.ceil(math.log2(n))

    print(f"{n:>15,} | {linear_steps:>18,} | {binary_steps:>18}")


In [ ]:

import matplotlib.pyplot as plt

sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
linear_steps = sizes
binary_steps = [math.ceil(math.log2(n)) for n in sizes]

plt.figure(figsize=(8, 5))
plt.plot(sizes, linear_steps, marker="o", label="Linear Search O(n)")
plt.plot(sizes, binary_steps, marker="o", label="Binary Search O(log n)")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Jumlah Data (log scale)")
plt.ylabel("Perkiraan Jumlah Langkah (log scale)")
plt.title("Pertumbuhan Linear Search vs Binary Search")
plt.legend()
plt.grid(True)
plt.show()



# Studi Kasus

Bagian berikut tidak hanya meminta implementasi, tetapi juga menganalisis apakah Binary Search merupakan pilihan yang tepat.



## 5. Studi Kasus 1 — Pencarian Mahasiswa Berdasarkan NIM

Sistem akademik memiliki data mahasiswa yang diurutkan berdasarkan NIM.

Tujuan:

> Cari mahasiswa dengan NIM `241007`.


In [ ]:

mahasiswa = [
    (241001, "Andi"),
    (241002, "Budi"),
    (241003, "Citra"),
    (241004, "Dinda"),
    (241005, "Eka"),
    (241006, "Fajar"),
    (241007, "Gita"),
    (241008, "Hendra"),
    (241009, "Intan"),
]


def cari_mahasiswa_by_nim(data, target_nim):
    low = 0
    high = len(data) - 1

    while low <= high:
        mid = (low + high) // 2
        nim, nama = data[mid]

        if nim == target_nim:
            return data[mid]

        elif nim < target_nim:
            low = mid + 1

        else:
            high = mid - 1

    return None


hasil = cari_mahasiswa_by_nim(mahasiswa, 241007)
print(hasil)



### Analisis Kasus 1

Jawab sebelum melanjutkan:

1. Mengapa Binary Search dapat digunakan pada data di atas?
2. Apa yang terjadi jika data mahasiswa diurutkan berdasarkan **nama**, tetapi pencarian dilakukan berdasarkan **NIM**?
3. Jika hanya ada 10 mahasiswa, apakah keuntungan Binary Search akan terasa signifikan?
4. Jika ada 1 juta mahasiswa dan pencarian dilakukan ribuan kali per hari, bagaimana pengaruhnya?



## 6. Studi Kasus 2 — Inventory SKU Produk

Sebuah gudang memiliki daftar SKU yang sudah terurut.

Tujuan:

> Cek apakah SKU `10037` tersedia.


In [ ]:

sku_produk = [
    10001, 10005, 10009, 10015,
    10021, 10037, 10045, 10051,
    10072, 10090
]

target_sku = 10037

index = binary_search(sku_produk, target_sku)

if index != -1:
    print(f"SKU {target_sku} ditemukan pada index {index}")
else:
    print(f"SKU {target_sku} tidak ditemukan")



### Twist

Misalkan data SKU masuk secara acak setiap detik:

```text
10021, 10005, 10090, 10001, 10037, ...
```

Pertanyaan:

- Apakah Binary Search bisa langsung digunakan?
- Jika kita selalu melakukan `sort()` sebelum setiap pencarian, apakah itu efisien?
- Bagaimana jika pencarian sangat jarang, tetapi penambahan data sangat sering?

> **Pelajaran:** algoritma terbaik bergantung pada pola penggunaan data, bukan hanya kompleksitas pencariannya.



## 7. Studi Kasus 3 — Data Sensor Berdasarkan Timestamp

Sensor IoT mengirim data setiap 60 detik. Data disimpan berdasarkan timestamp.

Tujuan pertama:

> Cari pembacaan sensor tepat pada timestamp tertentu.


In [ ]:

sensor_data = [
    (1710000000, 27.4),
    (1710000060, 27.6),
    (1710000120, 27.7),
    (1710000180, 27.9),
    (1710000240, 28.1),
    (1710000300, 28.0),
]


def cari_sensor_by_timestamp(data, target_timestamp):
    low = 0
    high = len(data) - 1

    while low <= high:
        mid = (low + high) // 2
        timestamp, nilai = data[mid]

        if timestamp == target_timestamp:
            return data[mid]

        elif timestamp < target_timestamp:
            low = mid + 1

        else:
            high = mid - 1

    return None


print(cari_sensor_by_timestamp(sensor_data, 1710000180))



### Pendalaman: bagaimana jika timestamp persisnya tidak ada?

Misalnya user mencari:

```text
1710000150
```

Timestamp tersebut berada di antara:

```text
1710000120
1710000180
```

Sekarang tugas kita berubah:

> Cari data sensor dengan timestamp yang **paling dekat** dengan target.

Ini masih dapat diselesaikan dengan ide Binary Search.


In [ ]:

def cari_timestamp_terdekat(data, target):
    low = 0
    high = len(data) - 1

    if not data:
        return None

    while low <= high:
        mid = (low + high) // 2
        timestamp = data[mid][0]

        if timestamp == target:
            return data[mid]

        elif timestamp < target:
            low = mid + 1

        else:
            high = mid - 1

    # Setelah loop:
    # high = kandidat di kiri
    # low  = kandidat di kanan
    candidates = []

    if 0 <= high < len(data):
        candidates.append(data[high])

    if 0 <= low < len(data):
        candidates.append(data[low])

    return min(candidates, key=lambda item: abs(item[0] - target))


target = 1710000150
print(cari_timestamp_terdekat(sensor_data, target))



## 8. Studi Kasus 4 — Budget: Cari Harga Terbesar yang Masih Terjangkau

Daftar harga sudah terurut:

```text
10.000
15.000
20.000
25.000
30.000
40.000
```

Budget user adalah:

```text
27.000
```

Target kita **bukan** mencari `27.000`, karena nilai tersebut tidak ada.

Targetnya:

> Cari harga terbesar yang masih `<= 27.000`.

Jawaban yang diharapkan adalah `25.000`.


In [ ]:

def harga_maksimal_sesuai_budget(harga, budget):
    low = 0
    high = len(harga) - 1
    best = None

    while low <= high:
        mid = (low + high) // 2

        if harga[mid] <= budget:
            best = harga[mid]
            low = mid + 1
        else:
            high = mid - 1

    return best


harga = [10_000, 15_000, 20_000, 25_000, 30_000, 40_000]

print(harga_maksimal_sesuai_budget(harga, 27_000))



### Insight

Binary Search tidak selalu mencari:

```python
data[mid] == target
```

Binary Search juga dapat digunakan untuk menemukan **batas**:

- nilai pertama yang `>= target`
- nilai terakhir yang `<= target`
- posisi penyisipan
- nilai terdekat
- batas minimum/maksimum yang memenuhi suatu kondisi



## 9. Python `bisect`

Python memiliki modul bawaan bernama `bisect` untuk bekerja dengan list terurut.

> Gunakan `bisect` setelah memahami konsep Binary Search, bukan sebagai pengganti pemahaman algoritmanya.


In [ ]:

import bisect

data = [10, 20, 30, 40, 50]

posisi = bisect.bisect_left(data, 35)

print("Posisi penyisipan:", posisi)

bisect.insort(data, 35)

print("Setelah disisipkan:", data)



### `bisect_left` vs `bisect_right`

Jika ada data duplikat:

```python
[10, 20, 20, 20, 30]
```

- `bisect_left()` → posisi paling kiri
- `bisect_right()` → posisi setelah kemunculan paling kanan


In [ ]:

data = [10, 20, 20, 20, 30]

print("bisect_left(20) :", bisect.bisect_left(data, 20))
print("bisect_right(20):", bisect.bisect_right(data, 20))



## 10. Kapan Binary Search Tidak Cocok?

Binary Search **bukan selalu jawaban terbaik**.

Pertimbangkan kembali Binary Search jika:

1. Data belum terurut dan hanya akan dicari satu kali.
2. Data berubah sangat sering sehingga menjaga urutannya mahal.
3. Struktur data tidak mendukung akses elemen tengah secara efisien.
4. Jumlah data sangat kecil sehingga perbedaan performa tidak relevan.
5. Kebutuhan pencarian lebih cocok menggunakan struktur lain seperti hash table/dictionary.

Contoh:

```python
data = [81, 12, 50, 7, 100, 31]
```

Kita tidak boleh langsung menerapkan Binary Search pada data tersebut.



# Deep Dive

## 11. Lower Bound

**Lower Bound** adalah posisi elemen pertama yang nilainya **lebih besar atau sama dengan target** (`>= target`).

Contoh:

```text
Data   : [10, 20, 30, 40, 50]
Target : 35
```

Lower bound berada pada nilai `40`.


In [ ]:

def lower_bound(data, target):
    low = 0
    high = len(data)

    while low < high:
        mid = (low + high) // 2

        if data[mid] < target:
            low = mid + 1
        else:
            high = mid

    return low


data = [10, 20, 30, 40, 50]

index = lower_bound(data, 35)

print("Index lower bound:", index)

if index < len(data):
    print("Nilai lower bound:", data[index])
else:
    print("Tidak ada nilai >= target")



## 12. Bonus — Binary Search on Answer

Ini merupakan penggunaan Binary Search yang lebih lanjut.

Binary Search tidak harus mencari elemen di dalam list.

Kita dapat mencari **nilai jawaban minimum/maksimum** selama terdapat kondisi yang bersifat monoton.

Contoh kasus:

> Sebuah server harus memproses `100.000` request dalam maksimal `8` detik.  
> Berapa **kapasitas minimum request/detik** yang dibutuhkan?

Jika kapasitas terlalu kecil → gagal memenuhi batas waktu.  
Jika kapasitas cukup besar → berhasil.

Polanya menjadi:

```text
False False False False True True True True ...
```

Kita dapat mencari titik `True` pertama dengan Binary Search.


In [ ]:

import math

total_request = 100_000
maks_waktu = 8


def cukup_cepat(kapasitas_per_detik):
    waktu_dibutuhkan = math.ceil(total_request / kapasitas_per_detik)
    return waktu_dibutuhkan <= maks_waktu


def kapasitas_minimum():
    low = 1
    high = total_request
    answer = high

    while low <= high:
        mid = (low + high) // 2

        if cukup_cepat(mid):
            answer = mid
            high = mid - 1
        else:
            low = mid + 1

    return answer


hasil = kapasitas_minimum()

print("Kapasitas minimum:", hasil, "request/detik")
print("Waktu yang dibutuhkan:", math.ceil(total_request / hasil), "detik")



## 13. Case Method — Pilih Algoritma yang Tepat

### Kasus

Sebuah aplikasi menyimpan **500.000 transaksi** berdasarkan urutan waktu masuk.

Setiap transaksi memiliki:

```text
transaction_id
timestamp
customer_id
amount
```

Data saat ini tersusun berdasarkan `timestamp`.

User ingin mencari transaksi berdasarkan `transaction_id`.

### Diskusikan

Jawab pertanyaan berikut:

1. Apakah Binary Search dapat langsung digunakan untuk mencari `transaction_id`?
2. Key apa yang saat ini membuat data terurut?
3. Jika kita melakukan sorting berdasarkan `transaction_id`, apa konsekuensinya?
4. Jika pencarian berdasarkan `transaction_id` dilakukan ribuan kali per hari, strategi apa yang masuk akal?
5. Apakah Linear Search, Binary Search, atau struktur data lain lebih tepat?
6. Jelaskan alasan pemilihan algoritma/struktur data Anda.

> Fokus penilaian: **analisis karakteristik masalah dan argumentasi**, bukan sekadar menuliskan kode.



## 14. Challenge 1 — Implementasikan Binary Search Sendiri

Lengkapi fungsi berikut tanpa melihat implementasi sebelumnya.


In [ ]:

def binary_search_challenge(data, target):
    # TODO:
    # 1. Tentukan low
    # 2. Tentukan high
    # 3. Buat perulangan
    # 4. Hitung mid
    # 5. Bandingkan data[mid] dengan target
    # 6. Update low/high
    # 7. Return index atau -1

    pass


# Test
data_test = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

# Harusnya menghasilkan index 5
# print(binary_search_challenge(data_test, 23))



## 15. Challenge 2 — Cari Nilai Pertama yang >= Target

Data:

```python
[5, 10, 10, 10, 17, 21, 30]
```

Target:

```text
10
```

Output yang diharapkan:

```text
index = 1
```

Buat sendiri versi *lower bound* menggunakan Binary Search.


In [ ]:

def first_greater_or_equal(data, target):
    # TODO: implementasikan dengan Binary Search
    pass


data_test = [5, 10, 10, 10, 17, 21, 30]

# print(first_greater_or_equal(data_test, 10))



## 16. Challenge 3 — Apakah Binary Search Tepat?

Untuk setiap skenario berikut, jawab:

- **Cocok / Tidak langsung cocok / Tidak cocok**
- Jelaskan alasannya.

### A
10 juta nomor pelanggan sudah terurut. Pencarian dilakukan ribuan kali per menit.

### B
50 nama peserta disimpan secara acak dan hanya dicari satu kali.

### C
Data log disimpan berdasarkan timestamp. Kita ingin mencari log pada waktu tertentu.

### D
Daftar produk disusun berdasarkan harga, tetapi user ingin mencari berdasarkan nama produk.

### E
Sensor menghasilkan data setiap detik dan user ingin mencari nilai pada timestamp tertentu.



# 17. Ringkasan

### Binary Search

- membutuhkan ruang pencarian yang **terurut sesuai key pencarian**;
- membagi ruang pencarian menjadi dua setiap langkah;
- memiliki kompleksitas waktu `O(log n)`;
- sangat efisien untuk data besar dan pencarian berulang.

### Tetapi...

Binary Search bukan hanya:

> "mencari satu angka di dalam array."

Binary Search dapat digunakan untuk:

- exact search;
- nearest value;
- lower bound;
- upper bound;
- insertion position;
- mencari batas minimum/maksimum;
- *Binary Search on Answer*.

---

## Pertanyaan Refleksi

1. Mengapa data terurut penting untuk Binary Search?
2. Mengapa Binary Search memiliki kompleksitas `O(log n)`?
3. Apa perbedaan terbesar antara Linear Search dan Binary Search?
4. Berikan satu contoh kasus ketika Binary Search **tidak layak** digunakan.
5. Menurut Anda, insight apa yang paling penting dari notebook ini?

> **Key takeaway:**  
> *Algoritma yang baik bukan hanya algoritma yang cepat, tetapi algoritma yang sesuai dengan karakteristik masalah dan data.*
